# 環境変数

In [ ]:
import os
os.environ["ERG_DATA_DIR"] = "/mnt/j/observation_data/"

# 3dfluxデータを時間軸に焼き直す

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np
import xarray as xr

pt.del_data('*')

time_range_full = ['2017-11-15/16:00:00', '2017-11-15/17:00:00']
psp.erg.lepi(time_range_full, datatype='3dflux', get_support_data=True, no_update=True, version='v03_00')

# 解析対象の狭い時間窓
time_range = ['2017-11-15/16:10:00', '2017-11-15/16:25:00']

# flux3d本体 (dims = time, v1(energy), v2(channel), v3(phase))  # [#/cm2/sr/sec/keV]
flux3d = pt.data_quants['erg_lepi_l2_3dflux_FPDU'].sel(
    time=slice(*time_range)
)

# 各軸の座標を取り出しておく
time_ax     = flux3d.time.values        # shape = (T,)
energy_ax   = flux3d.v1.values          # (E,)
channel_ax  = flux3d.v2.values          # (C,)
spin_ax     = flux3d.v3.values          # (S,)

time_num, energy_num, channel_num, spin_num = len(time_ax), len(energy_ax), len(channel_ax), len(spin_ax)   # T, E, C, S

# 1 spin time = 8 sec
# 1 spin phase time = 0.5 sec
# 1 energy time step = 15625 μsec
spin_offset_ns      = np.arange(spin_num, dtype='timedelta64[ns]') * 500_000_000        # 0.5 sec = 500,000,000 nsec
energy_offset_ns    = (np.arange(energy_num, dtype='int64') * 15_625_000 + 7_812_500).astype('timedelta64[ns]')

offset_ns           = energy_offset_ns[:, None] + spin_offset_ns    # (E, 1) + (S) -> (E, S)

flux_E_TS_C = flux3d.transpose('v1_dim', 'time', 'v3_dim', 'v2_dim').values.reshape(energy_num, time_num*spin_num, channel_num)

# xarray.DataArrayをエネルギーごとに生成
flux_data_arrays = {}
for energy_i in range(energy_num):
    time_flat = (time_ax[:, None] + offset_ns[energy_i][None, :]).reshape(-1) # ((T, 1) + (1, S) -> (T, S)).reshape(-1) -> (TxS,)

    flux_data_arrays[energy_i] = xr.DataArray(
        flux_E_TS_C[energy_i],
        dims=['time', 'channel'],
        coords={
            'time':         time_flat,
            'channel':      channel_ax,
            'energy_keV':   energy_ax[energy_i]
        },
        attrs=flux3d.attrs,
        name=f'flux_energy_{energy_i}'
    )

print(flux_data_arrays)

fidu_angle_dict     = pt.data_quants['erg_lepi_l2_3dflux_FIDU_Angle_sga']
fidu_angle  = fidu_angle_dict['data'].astype(float)     # shape (2, 3, 16)
AZ_deg_mid = fidu_angle[0, 1, :channel_num]      # shape (C,)
# AZ_deg_mid =  [ 78.75  56.25  33.75  11.25 -11.25 -33.75 -56.25 -78.75]

theta_sga = AZ_deg_mid                  # (C,)
varphi_sga = -90. * np.ones(spin_num)   # (S,)

# (TxS, C)の2次元配列を生成
theta_sga_time = np.tile(theta_sga, (time_num*spin_num, 1))   # (TxS, C)
varphi_sga_times = np.tile(varphi_sga, (time_num, 1)).reshape(-1, 1)   # (TxS, 1)
varphi_sga_time = np.tile(varphi_sga_times, (1, channel_num))   # (TxS, C)

angle_sga_time = np.stack((theta_sga_time, varphi_sga_time), axis=2)   # (TxS, C, 2)

# theta_sga, varphi_sga -> Vx_sga, Vy_sga, Vz_sga
vector_sga_time = np.zeros((time_num*spin_num, channel_num, 3))   # (TxS, C, 3)
vector_sga_time[:, :, 0] = np.cos(np.radians(angle_sga_time[:, :, 0])) * np.cos(np.radians(angle_sga_time[:, :, 1]))
vector_sga_time[:, :, 1] = np.cos(np.radians(angle_sga_time[:, :, 0])) * np.sin(np.radians(angle_sga_time[:, :, 1]))
vector_sga_time[:, :, 2] = np.sin(np.radians(angle_sga_time[:, :, 0]))

v_unit_vector_sga_energy_channel_list = {}
for energy_i in range(energy_num):
    time_flat = (time_ax[:, None] + offset_ns[energy_i][None, :]).reshape(-1)

    for channel_i in range(channel_num):
        v_unit_vector_sga_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            vector_sga_time[:, channel_i, :],
            dims=['time', 'xyz'],
            coords={
                'time': time_flat,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'v_unit_vector_sga_{energy_i}_{channel_i}'
        )
        dot_ = (v_unit_vector_sga_energy_channel_list[energy_i, channel_i] * v_unit_vector_sga_energy_channel_list[energy_i, channel_i]).sum(dim='xyz')
        print(f'v_unit_vector_sga_energy_channel_list[{energy_i}, {channel_i}] = ', np.nanmin(dot_), np.nanmax(dot_), np.nanmean(dot_))

In [ ]:
import pyspedas as psp
import pytplot as pt
import xarray as xr

v_unit_vector_sgi_energy_channel_list = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        pt.store_data(f'vector_sga_{energy_i}_{channel_i}', data={'x': v_unit_vector_sga_energy_channel_list[energy_i, channel_i].time, 'y': v_unit_vector_sga_energy_channel_list[energy_i, channel_i].values})
        # SGI座標系に変換
        psp.projects.erg.sga2sgi(name_in=f'vector_sga_{energy_i}_{channel_i}', name_out=f'vector_sgi_{energy_i}_{channel_i}')
        _data   = pt.data_quants[f'vector_sgi_{energy_i}_{channel_i}'].rename({"v_dim": "xyz"})
        #_data_2 = (_data * _data).sum(dim='xyz')
        _data_unit  = _data #/ np.sqrt(_data_2)
        v_unit_vector_sgi_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            _data_unit.data,
            dims=['time', 'xyz'],
            coords={
                'time': _data.time,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'v_unit_vector_sgi_{energy_i}_{channel_i}'
        )

for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        _dot    = (v_unit_vector_sgi_energy_channel_list[energy_i, channel_i] * v_unit_vector_sgi_energy_channel_list[energy_i, channel_i]).sum(dim='xyz')
        print(f'v_unit_vector_sgi_energy_channel_list[{energy_i}, {channel_i}] = ', np.nanmin(_dot), np.nanmax(_dot), np.nanmean(_dot))

In [ ]:
import pyspedas as psp
import pytplot as pt
import xarray as xr

v_unit_vector_dsi_energy_channel_list = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        # dsi座標系に変換
        psp.projects.erg.sgi2dsi(name_in=f'vector_sgi_{energy_i}_{channel_i}', name_out=f'vector_dsi_{energy_i}_{channel_i}')
        _data   = pt.data_quants[f'vector_dsi_{energy_i}_{channel_i}'].rename({"v_dim": "xyz"})
        #_data_2 = (_data * _data).sum(dim='xyz')
        _data_unit  = _data #/ np.sqrt(_data_2)
        v_unit_vector_dsi_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            _data_unit.data,
            dims=['time', 'xyz'],
            coords={
                'time': _data.time,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'v_unit_vector_dsi_{energy_i}_{channel_i}'
        )

for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        dot_    = (v_unit_vector_dsi_energy_channel_list[energy_i, channel_i] * v_unit_vector_dsi_energy_channel_list[energy_i, channel_i]).sum(dim='xyz')
        print(f'v_unit_vector_dsi_energy_channel_list[{energy_i}, {channel_i}] = ', np.nanmin(dot_), np.nanmax(dot_), np.nanmean(dot_))

# 背景磁場ベクトルの決定

In [ ]:
import pyspedas as psp
import pytplot as pt
import xarray as xr
import numpy as np

psp.erg.mgf(trange=time_range_full, level='l2', datatype='256hz', coord='dsi', version='v03.03', no_update=True)
psp.erg.mgf(trange=time_range_full, level='l2', datatype='64hz', coord='dsi', version='v03.03', no_update=True)
psp.erg.mgf(trange=time_range_full, level='l2', datatype='8sec', coord='dsi', version='v03.03', no_update=True)

B_256Hz = pt.data_quants['erg_mgf_l2_mag_256hz_dsi'].rename({"v_dim": "xyz"})
B_64Hz = pt.data_quants['erg_mgf_l2_mag_64hz_dsi'].rename({"v_dim": "xyz"})
B_8sec  = pt.data_quants['erg_mgf_l2_mag_8sec_dsi'].rename({"v_dim": "xyz"})

In [ ]:
background_time_sec = 100 #[sec]

In [ ]:
B0_vector_dsi_energy_channel_list = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        B_256Hz_interp       = B_256Hz.interp(time=v_unit_vector_dsi_energy_channel_list[energy_i, channel_i].time, method='linear')
        dt_B_256Hz_interp    = (B_256Hz_interp.time[1] - B_256Hz_interp.time[0]) / np.timedelta64(1, 's')
        B_background        = B_256Hz_interp.rolling(time=int(background_time_sec/dt_B_256Hz_interp), center=True).mean('time')

        B0_vector_dsi_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            B_background.data,
            dims=['time', 'xyz'],
            coords={
                'time': B_background.time,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'B0_vector_dsi_{energy_i}_{channel_i}'
        )
        print(f'B0_vector_dsi_energy_channel_list[{energy_i}, {channel_i}] = ', B0_vector_dsi_energy_channel_list[energy_i, channel_i])

# 対象とする垂直磁場成分を取得

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, sosfreqz

# ------------------ フィルタ設計 ------------------
fs = 64.                  # サンプリング周波数 [Hz]
order = 4                   # 4次 Butterworth（2-pole × 2-stage）

# 0.6 Hzから0.75 Hzの範囲のみを通すband-passフィルタ
lowcut = 0.45
highcut = 0.75

# btypeを'bandpass'に設定
sos = butter(N=order, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')

# ------------------ インパルス応答 (変更なし) ------------------
n = 2048
delta = np.zeros(n)
delta[n//2] = 1

h = sosfiltfilt(sos, delta)
t = (np.arange(n) - n//2) / fs

# ------------------ 周波数応答 (変更なし) ------------------
w, H = sosfreqz(sos, worN=4096, fs=fs)
H_dbl = np.abs(H)**2

# ------------------ プロット (タイトルとハイライトを変更) ------------------
fig, axs = plt.subplots(2, 1, figsize=(10, 6), tight_layout=True)

# 時間領域
axs[0].plot(t, h)
axs[0].set_title('Impulse Response (Band-pass filter)')
axs[0].set_xlabel('Time [s]')
axs[0].set_ylabel('Amplitude')
axs[0].grid(True)

# 周波数領域
axs[1].semilogx(w, 20*np.log10(H_dbl), label='|H(f)|')

# 除去帯域を半透明のグレーで示す
axs[1].axvspan(lowcut, highcut, color='gray', alpha=0.3, label=f'{lowcut:.2f}-{highcut:.2f} Hz')
axs[1].axvline(0.6, color='red', lw=1)
axs[1].axvline(0.75, color='red', lw=1)

axs[1].set_title('Magnitude Response (Band-pass filter)')
axs[1].set_xlabel('Frequency [Hz]')
axs[1].set_ylabel('Magnitude [dB]')
axs[1].set_ylim(-20, 10)
axs[1].set_xlim(3E-1, 1)
axs[1].legend()
axs[1].grid(True, which='both', ls='--')

plt.show()

In [ ]:
t_B_8sec    = B_8sec.time
dt_B_8sec = (t_B_8sec[2] - t_B_8sec[1]) / np.timedelta64(1, 's')
B_background    = B_8sec.rolling(time=int(background_time_sec/dt_B_8sec), center=True).mean('time')

B_background_64Hz  = B_background.interp(time=B_64Hz.time, method='linear')
B_64Hz_perturb     = B_64Hz - B_background_64Hz
B_64Hz_perp = B_64Hz_perturb - (B_64Hz_perturb * B_background_64Hz).sum(dim='xyz') / (B_background_64Hz * B_background_64Hz).sum(dim='xyz') * B_background_64Hz

In [ ]:
import numpy as np
from scipy.signal import butter, sosfiltfilt
import pytplot as pt
import matplotlib.pyplot as plt
import os # osモジュールもインポートしておく

# フィルタパラメータ
fs = 64.                  # サンプリング周波数 [Hz]
lowcut = 0.60              # 通過域の下限周波数 [Hz]
highcut = 0.75            # 通過域の上限周波数 [Hz]
order = 4                 # フィルタの次数
window_sec = background_time_sec        # 背景磁場の移動平均窓幅 [sec]

sos = butter(N=order, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')

def apply_filter_segmented(y, sos_mat):
    """NaN を含む 1‑D 配列にセグメントごとで sosfiltfilt を適用する"""
    good = np.isfinite(y)
    out  = np.full_like(y, np.nan)
    idx  = np.where(good)[0]
    segs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
    
    # パディング長はフィルタの次数に依存
    # scipyのドキュメントによると、sosfiltfiltのデフォルトpadlenは 3 * (sos.shape[1] // 2 - 1)
    # sosの形状は (n_sections, 6) なので、padlenは 3 * 2 = 6 となる
    padlen = 3 * (sos_mat.shape[1] - 1)
    
    for s in segs:
        if s.size > padlen:
            out[s] = sosfiltfilt(sos_mat, y[s])
    return out

B_64Hz_perp_bandpass = np.zeros(B_64Hz_perp.data.shape) * np.nan  # NaNで初期化
for i in range(3):
    B_64Hz_perp_bandpass[:, i] = apply_filter_segmented(B_64Hz_perp.data[:, i], sos)
da_B_64Hz_perp_bandpass = xr.DataArray(
    B_64Hz_perp_bandpass,
    dims=B_64Hz_perp.dims,
    coords=B_64Hz_perp.coords,
    name='B_64Hz_perp_bandpass'
)

da_B_64Hz_perp_bandpass_amp = np.sqrt((da_B_64Hz_perp_bandpass * da_B_64Hz_perp_bandpass).sum(dim='xyz'))

In [ ]:
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

time_ax_range_min = np.datetime64('2017-11-15T16:17:00')
time_ax_range_max = np.datetime64('2017-11-15T16:22:00')

# plot
fig, ax = plt.subplots(1, 1, figsize=(10, 4), sharex=True)
ax.plot(da_B_64Hz_perp_bandpass_amp.time, da_B_64Hz_perp_bandpass_amp.data, lw=0.5, c='k')
ax.set_ylabel('[nT]')
ax.minorticks_on()
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.set_xlim(time_ax_range_min, time_ax_range_max)
ax.set_ylim(0, 10)
plt.tight_layout()
plt.show()

# v_unitとB0、B_perpとのなす角を求めて、pitch angleとzeta angleをfluxデータに付与

In [ ]:
def ensure_xyz_coord(da):
    if 'xyz' in da.dims and 'xyz' not in da.coords:
        da = da.assign_coords(xyz=['x','y','z'])
    return da

flux_pitch_zeta_data_list    = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        v_unit  = v_unit_vector_dsi_energy_channel_list[energy_i, channel_i]
        B0 = B0_vector_dsi_energy_channel_list[energy_i, channel_i].interp(time=v_unit.time)
        Bperp = da_B_64Hz_perp_bandpass.interp(time=v_unit.time)
        Bperp = Bperp - (Bperp * B0).sum(dim='xyz') / (B0 * B0).sum(dim='xyz') * B0

        v_unit  = ensure_xyz_coord(v_unit)
        B0      = ensure_xyz_coord(B0)
        Bperp   = ensure_xyz_coord(Bperp)

        dot_vB0 = (v_unit * B0).sum(dim='xyz')
        B0_2    = (B0 * B0).sum(dim='xyz')
        alpha = np.arccos(dot_vB0 / np.sqrt(B0_2))

        v_perp = v_unit - dot_vB0 / B0_2 * B0
        cross = xr.apply_ufunc(np.cross, Bperp, v_perp,
                               input_core_dims=[['xyz'], ['xyz']],
                               output_core_dims=[['xyz']], vectorize=True)
        Bperp_2     = (Bperp * Bperp).sum(dim='xyz')
        sin_zeta    = (cross * B0).sum(dim='xyz') / np.sqrt(B0_2 * Bperp_2)
        cos_zeta    = (v_perp * Bperp).sum(dim='xyz') / np.sqrt(Bperp_2)
        zeta = np.atan2(sin_zeta, cos_zeta)
        
        # 時間を統一（zeta基準）
        t = zeta['time']
        
        # flux の time を zeta に合わせる（必要なら補間）
        flux_ch = xr.DataArray(
            flux_data_arrays[energy_i][:, channel_i],
            coords={'time': flux_data_arrays[energy_i].coords['time']},  # ここは実データのtimeに合わせる
            dims=('time',)
        ).interp(time=t)
        
        da = xr.concat(
            [
                flux_ch.rename('differential_number_flux_keV'),
                np.rad2deg(alpha).rename('pitch_angle_deg'),
                (np.rad2deg(zeta) % 360.0).rename('zeta_angle_deg')
            ],
            dim='variable'
        ).assign_coords(variable=['differential_number_flux_keV','pitch_angle_deg','zeta_angle_deg']) \
         .transpose('time','variable')
        
        flux_pitch_zeta_data_list[energy_i, channel_i] = da.assign_coords(
            channel=channel_ax[channel_i],
            energy_keV=energy_ax[energy_i],
        )
        print(flux_pitch_zeta_data_list[energy_i, channel_i])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

time_ax_range_min = np.datetime64('2017-11-15T16:17:00')
time_ax_range_max = np.datetime64('2017-11-15T16:22:00')

for energy_i in range(energy_num):
    if energy_i != 5:
        continue

    # --- 1) 全channelでvmin/vmaxを決める（>0 かつ有限のみ） ---
    flux_vals = []
    for channel_i in range(channel_num):
        d = flux_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        f = d.sel(variable='differential_number_flux_keV').values
        flux_vals.append(f.ravel())
    flux_all = np.concatenate(flux_vals)
    mask = np.isfinite(flux_all) & (flux_all > 0)
    if not mask.any():
        continue
    vmin, vmax = flux_all[mask].min(), flux_all[mask].max()
    if np.log10(vmin) < np.log10(vmax) -2:
        vmin = vmax*1E-2
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    cmap = 'turbo'

    # --- 2) 描画 ---
    fig = plt.figure(figsize=(10, 12))
    ax0 = fig.add_subplot(211)
    ax1 = fig.add_subplot(212)

    for channel_i in range(channel_num):
        #if channel_i != 0:
        #    continue
        d = flux_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        t = d['time'].values
        flux = d.sel(variable='differential_number_flux_keV').values
        alpha = d.sel(variable='pitch_angle_deg').values
        zeta  = d.sel(variable='zeta_angle_deg').values

        mask = ((alpha <= 11.25) | (alpha >= 167.75)) & (flux > 0)
        if np.any(mask):
            for ti, fi, ai, zi in zip(t[mask], flux[mask], alpha[mask], zeta[mask]):
                print(energy_i, channel_i, np.datetime_as_string(ti, unit='ns'), fi, ai, zi)

        ax0.scatter(t, alpha, c=flux, s=10, cmap=cmap, norm=norm, rasterized=True)
        ax1.scatter(t, zeta,  c=flux, s=10, cmap=cmap, norm=norm, rasterized=True)

    # 軸体裁
    ax0.set_ylabel(r'Pitch Angle $\alpha$' + '\n[deg]')
    ax1.set_ylabel(r'Phase difference $\zeta$' + '\n[deg]')

    ax0.set_title(f'LEP-i flux (energy = {energy_ax[energy_i]:.4f} keV)')
    for ax in (ax0, ax1):
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
        ax.minorticks_on()
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_xlim(time_ax_range_min, time_ax_range_max)
    ax0.set_ylim(0, 180);  ax0.set_yticks(np.arange(0, 181, 15))
    ax1.set_ylim(0, 360);  ax1.set_yticks(np.arange(0, 361, 30))

    # --- 3) カラーバーは共通norm/cmapから作る ---
    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])  # 必須
    plt.colorbar(sm, ax=ax0, label='Differential number flux\n' + r'[$\mathrm{s}^{-1}\mathrm{cm}^{-2}\mathrm{str}^{-1}\mathrm{keV}^{-1}$]')
    plt.colorbar(sm, ax=ax1, label='Differential number flux\n' + r'[$\mathrm{s}^{-1}\mathrm{cm}^{-2}\mathrm{str}^{-1}\mathrm{keV}^{-1}$]')

    plt.tight_layout()
    plt.show()


# Differential Number Flux $J$ をCount Number $C$に変換

In [ ]:
sampling_time   = 0.015625  # [sec]

In [ ]:
def G_Factor_func(energy):
    return  (1.52 - 0.108 * np.log10(energy)) * 1E-3  # [cm^-2 str keV keV^-1 channel^-1]

G_Factor_ax = G_Factor_func(energy_ax)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(8, 5), sharex=True)
ax.plot(energy_ax, G_Factor_ax, lw=2, c='red')
ax.set_xlabel(r'Energy per charge [$\mathrm{keV/q}$]')
ax.set_ylabel(r'Geometric factor ($G_{\mathrm{ESA}}$)' + '\n' + r'[$\mathrm{cm}^{-2} \, \mathrm{str} \, \mathrm{keV} \, \mathrm{keV}^{-1} \, \mathrm{channel}^{-1}$]')
ax.minorticks_on()
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.set_xscale('log')
ax.set_xlim(np.nanmin(energy_ax), np.nanmax(energy_ax))
ax.set_ylim(0.00135, 0.0018)
ax.axvline(0.01, c='b', linestyle=':', lw=2)
ax.axhline(0.00170, c='b', linestyle=':', lw=2)
ax.axvline(12, c='orange', linestyle=':', lw=2)
ax.axhline(0.00140, c='orange', linestyle=':', lw=2)
plt.tight_layout()
plt.show()

In [ ]:
Efficiency  = 0.7   # Asamura et al. (2018)でのdetection efficiency of STOP signalsを仮採用

In [ ]:
count_pitch_zeta_data_list    = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        data_ = flux_pitch_zeta_data_list[energy_i, channel_i]
        G_Factor_   = G_Factor_func(energy_ax[energy_i])

        count_ch = data_[:, 0] * G_Factor_ * Efficiency * sampling_time

        da = xr.concat(
            [
                count_ch.rename('count_number'),
                data_[:, 1],
                data_[:, 2]
            ],
            dim='variable'
        ).assign_coords(variable=['count_number','pitch_angle_deg','zeta_angle_deg']) \
         .transpose('time','variable')
        
        count_pitch_zeta_data_list[energy_i, channel_i] = da.assign_coords(
            channel=channel_ax[channel_i],
            energy_keV=energy_ax[energy_i],
        )
        print(count_pitch_zeta_data_list[energy_i, channel_i][900:930, :])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

time_ax_range_min = np.datetime64('2017-11-15T16:17:00')
time_ax_range_max = np.datetime64('2017-11-15T16:22:00')

for energy_i in range(energy_num):
    if energy_i != 5:
        continue

    # --- 1) 全channelでvmin/vmaxを決める（>0 かつ有限のみ） ---
    count_vals = []
    for channel_i in range(channel_num):
        d = count_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        f = d.sel(variable='count_number').values
        count_vals.append(f.ravel())
    count_all = np.concatenate(count_vals)
    mask = np.isfinite(count_all) & (count_all > 1)
    if not mask.any():
        continue
    vmin, vmax = count_all[mask].min(), count_all[mask].max()
    vmax = np.ceil(vmax)
    vmin = 1
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap = 'turbo'

    # --- 2) 描画 ---
    fig = plt.figure(figsize=(10, 12))
    ax0 = fig.add_subplot(211)
    ax1 = fig.add_subplot(212)

    for channel_i in range(channel_num):
        #if channel_i != 0:
        #    continue
        d = count_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        t = d['time'].values
        count = d.sel(variable='count_number').values
        alpha = d.sel(variable='pitch_angle_deg').values
        zeta  = d.sel(variable='zeta_angle_deg').values
        mask = np.isfinite(count) & (count > 1)
        t   = t[mask]
        count   = count[mask]
        alpha   = alpha[mask]
        zeta    = zeta[mask]
        ax0.scatter(t, alpha, c=count, s=10, cmap=cmap, norm=norm, rasterized=True)
        ax1.scatter(t, zeta,  c=count, s=10, cmap=cmap, norm=norm, rasterized=True)

    # 軸体裁
    ax0.set_ylabel(r'Pitch Angle $\alpha$' + '\n[deg]')
    ax1.set_ylabel(r'Phase difference $\zeta$' + '\n[deg]')

    ax0.set_title(f'LEP-i count number (energy = {energy_ax[energy_i]:.4f} keV)')
    for ax in (ax0, ax1):
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
        ax.minorticks_on()
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_xlim(time_ax_range_min, time_ax_range_max)
    ax0.set_ylim(0, 180);  ax0.set_yticks(np.arange(0, 181, 15))
    ax1.set_ylim(0, 360);  ax1.set_yticks(np.arange(0, 361, 30))

    # --- 3) カラーバーは共通norm/cmapから作る ---
    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])  # 必須
    plt.colorbar(sm, ax=ax0, label='Count Number')
    plt.colorbar(sm, ax=ax1, label='Count Number')

    plt.tight_layout()
    plt.show()


# 電場ベクトルの取得

In [ ]:
import pyspedas as psp
import pytplot as pt
import xarray as xr
import numpy as np

psp.erg.pwe_efd(trange=time_range_full, level='l2', datatype='64hz', coord='dsi', no_update=True)

In [ ]:
Ex_64Hz_dsi = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ex_waveform']
Ey_64Hz_dsi = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ey_waveform']

Ex_64Hz_dsi_interp  = Ex_64Hz_dsi.interp(time=B_background_64Hz.time)
Ey_64Hz_dsi_interp  = Ey_64Hz_dsi.interp(time=B_background_64Hz.time)

Ez_64Hz_dsi_interp  = xr.where(np.abs(B_background_64Hz[:, 2]) > 1E-12, - (Ex_64Hz_dsi_interp * B_background_64Hz[:, 0] + Ey_64Hz_dsi_interp * B_background_64Hz[:, 1]) / B_background_64Hz[:, 2], np.nan)

E_64Hz  = xr.Dataset({
    'E64_dsi_x':    Ex_64Hz_dsi_interp,
    'E64_dsi_y':    Ey_64Hz_dsi_interp,
    'E64_dsi_z':    Ez_64Hz_dsi_interp
})

E_64Hz